# The Hohmanns Transfer Orbit with "four-body" Problem
By Timmy Ly & Keaton Fitzgerald

In this project, we will introduce the Hohmann Transfer Orbit through the "four-body" problem by examining the three-body problem and incorporating orbits using a Python simulation. A basic definition of theHohmann Transfer Orbit is that it is the maneuver that transfers spacecraft from one orbit to another. The "three-body" problem is where three masses move and interact within their Newtonian gravitational potential, which can create an orbital movement between masses, or even chaos. From this basic definition, we try to simulate an environment where the sun will act as a center of mass with the Earth and the Asteroid orbiting it like a "three- body" problem. From the observation of the "three-body" problem, we will replace the asteroids with Mars to orbit the Sun, and also introduce a rocket launched from Earth that will move out of Earth’s orbit and enter Mars’ orbit. This project extends the classical "two-body" problem to a more realistic "four-body" scenario while using the Hohmann Transfer Orbit Theory. Our implementation combines numerical integration of gravitational interac-tions, and velocity changes maneuvers for Rocket Ship to enter the Mars’ Orbit.

In [ ]:
#Testing Case
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML

G=1.0 #Gravitational Constant
#Set up Experimental Masses (sun, earth, asteroids)
m_sun,m_earth,m_ast= 10.0, 0.5, 1e-05

#deriv function of three_body
def three_body(t,y):
    r1,r2,r3=y[0:2],y[2:4],y[4:6]
    v1,v2,v3=y[6:8],y[8:10],y[10:12]
    def accel(ri,rj,mj):
        diff=rj-ri
        return G*mj*diff/np.linalg.norm(diff)**3 #equation to calculate acceleration after (a = F/m)
    a1= np.zeros(2)
    a2=accel(r2,r1,m_sun)+accel(r2,r3,m_ast)
    a3=accel(r3,r1,m_sun)+accel(r3,r2,m_earth)
    return np.concatenate([v1,v2,v3,a1,a2,a3])

#(x1,y1,x2,y2,x3,y3.v1x,v1y,v2x,v2y,v3x,v3y)
y0=[0.0,0.0,7.5,0.0,15.0,0.0,0.0,0.0,0.0,0.5,-0.5,0.5] 
t_span=(0,600)
t_eval=np.linspace(*t_span,1000)
sol=solve_ivp(three_body,t_span,y0,t_eval=t_eval,rtol=1e-9,atol=1e-12)

#2D Plot
fig1,ax1=plt.subplots(figsize=(6,6))
ax1.plot(sol.y[0],sol.y[1],label='Sun')
ax1.plot(sol.y[2],sol.y[3],label='Earth')
ax1.plot(sol.y[4],sol.y[5],label='Asteroid')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.legend()
ax1.grid(True)
plt.show()

#3D Plot
fig3=plt.figure(figsize=(8,6))
ax3=fig3.add_subplot(projection='3d')
ax3.plot(sol.t,sol.y[0],sol.y[1],label='Sun')
ax3.plot(sol.t,sol.y[2],sol.y[3],label='Earth')
ax3.plot(sol.t,sol.y[4],sol.y[5],label='Asteroid')
ax3.set_xlabel('Time')
ax3.set_ylabel('x')
ax3.set_zlabel('y')
ax3.legend()
plt.show()

#2D animation
fig2,ax2=plt.subplots(figsize=(6,6))
ax2.set_xlim(sol.y[:6:2].min()*1.1,sol.y[:6:2].max()*1.1)
ax2.set_ylim(sol.y[1:6:2].min()*1.1,sol.y[1:6:2].max()*1.1)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Three-Body Motion')
ax2.grid(True)
lines=[ax2.plot([],[],lw=1)[0] for _ in range(3)]
points=[ax2.plot([],[],'o')[0] for _ in range(3)]
def init():
    for ln,pt in zip(lines,points):
        ln.set_data([],[])
        pt.set_data([],[])
    return lines+points
def update(frame):
    for i in range(3):
        xi,yi=sol.y[2*i],sol.y[2*i+1]
        lines[i].set_data(xi[:frame],yi[:frame])
        points[i].set_data([xi[frame-1]],[yi[frame-1]])
    return lines+points
ani=FuncAnimation(fig2,update,frames=len(t_eval),init_func=init,blit=True,interval=20)
HTML(ani.to_jshtml())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from mpl_toolkits.mplot3d import Axes3D          
from scipy.integrate import solve_ivp


G          = 1.0 # gravitational constant
r_earth    = 5.0  # Earth orbital radius
r_mars     = 15.0  # Mars orbital radius
M_sun, M_earth, M_mars, M_ship = 10.0, 0.5, .05, 1e-08 #ship is "massless" comparing to planets and the Sun

# Equation of motion (Newtonian Gravity) on bodies, affected by the Sun
def two_body(t, y):
    x0, y0, vx, vy = y
    # r^3
    r3 = (x0*x0 + y0*y0)**1.5               
    ax = -G * M_sun * x0 / r3
    ay = -G * M_sun * y0 / r3
    return [vx, vy, ax, ay]

# stop integration when spacecraft radius = r_mars (align ship with Mars)
def mars_orbit_event(t, y):
    return np.hypot(y[0], y[1]) - r_mars
mars_orbit_event.terminal  = True
mars_orbit_event.direction = 1

# Hohmann transfer 
a_transfer = 0.5 * (r_earth + r_mars)  #semi major axis of the transfer ellipse
v_ship     = np.sqrt(G * M_sun * (2/r_earth - 1/a_transfer)) #velocity of ship when transferring

# half‑period of the Keplerian orbit (Kepler's 3rd Law)   
t_half = np.pi * np.sqrt(a_transfer**3 / (G * M_sun))

# integrate spacecraft trajectory
y0_ship = [r_earth, 0.0, 0.0, v_ship]    # [x, y, vx, vy]
sol   = solve_ivp(two_body, (0, t_half), y0_ship, events=mars_orbit_event, max_step=0.02) #solve_ivp with range (0, t_half)
t_arrival = sol.t[-1] #time ship arrive at Mars
x_ship, y_ship = sol.y[0], sol.y[1] #postion of the ship

omega_e = np.sqrt(G * M_sun / r_earth**3)  # angular velocity for Earth
omega_m = np.sqrt(G * M_sun / r_mars**3)   #angular velocity for Mars
phi_m0 = (np.pi - omega_m * t_arrival) % (2*np.pi) # choose Mars phase to meet ship

#2D Plot
phi = np.linspace(0, 2*np.pi, 400) 
fig2 = plt.figure(figsize=(6,6))
ax2  = fig2.add_subplot(111)

ax2.plot(r_earth*np.cos(phi), r_earth*np.sin(phi),
         color='royalblue', label='Earth Orbit') # polar coordinates position of Earth
ax2.plot(r_mars*np.cos(phi),  r_mars*np.sin(phi),
         color='tomato', label='Mars Orbit') # polar coordinates position of Mars
ax2.plot(x_ship, y_ship, ':k', label='Transfer Path') #Plot position of the ship
ax2.scatter(0,0, s=120, c='gold', edgecolor='k', zorder=5, label='Sun') #create plot for the Sun
ax2.set_aspect('equal')
ax2.set_xlabel('x (arbitrary units)')
ax2.set_ylabel('y (arbitrary units)')
ax2.set_title('2‑D Orbits and Hohmann Transfer')
ax2.legend();  ax2.grid(ls=':')

#3D Plot
fig3 = plt.figure(figsize=(6,4))
ax3  = fig3.add_subplot(111, projection='3d')
ax3.plot(r_earth*np.cos(phi), r_earth*np.sin(phi), 0, color='royalblue') #earth polar coords plot
ax3.plot(r_mars*np.cos(phi),  r_mars*np.sin(phi), 0, color='tomato') #Mars polar coords plot
ax3.plot(x_ship, y_ship, np.zeros_like(x_ship), ':k') #ship plot 
ax3.scatter(0,0,0, c='gold', s=100, edgecolor='k') #The sun at origin in x, y, z plane
ax3.set(title='3‑D view (orbital plane z=0)', xlabel='x', ylabel='y', zlabel='z')
plt.tight_layout()

#2D Animation
N_FRAMES = 80   # can adjust this to increase the frame                              
idx      = np.linspace(0, len(x_ship)-1, N_FRAMES, dtype=int)
t_anim   = sol.t[idx]
x_ship, y_ship = x_ship[idx], y_ship[idx]
x_e = r_earth * np.cos(omega_e * t_anim) #earth position in x
y_e = r_earth * np.sin(omega_e * t_anim) #earth position in y
x_m = r_mars  * np.cos(omega_m * t_anim + phi_m0) #Mars' position in x
y_m = r_mars  * np.sin(omega_m * t_anim + phi_m0) #Mars' position in y

figA, axA = plt.subplots(figsize=(3.5,3.5))
axA.set_aspect('equal')
axA.set_xlim(-r_mars*1.1, r_mars*1.1)
axA.set_ylim(-r_mars*1.1, r_mars*1.1)
axA.axis('off')

# orbits and the Sun
axA.plot(r_earth*np.cos(phi), r_earth*np.sin(phi), color='royalblue', lw=0.8, alpha=0.6)
axA.plot(r_mars*np.cos(phi),  r_mars*np.sin(phi), color='tomato',    lw=0.8, alpha=0.6)
axA.plot(0,0, marker='o', color='gold', markersize=5, zorder=5)

#color and lines for animation
ln_ship,   = axA.plot([], [], ':k', lw=0.8)
pt_ship,   = axA.plot([], [], marker='*', color='k', markersize=4, zorder=6)
pt_e,    = axA.plot([], [], marker='o', color='royalblue', markersize=3)
pt_m,    = axA.plot([], [], marker='o', color='tomato',    markersize=3)

#animation code
def init_anim():
    ln_ship.set_data([], []);  pt_ship.set_data([], [])
    pt_e.set_data([], []);   pt_m.set_data([], [])
    return ln_ship, pt_ship, pt_e, pt_m

def update_anim(i):
    ln_ship.set_data(x_ship[:i+1], y_ship[:i+1])
    pt_ship.set_data(x_ship[i], y_ship[i])
    pt_e.set_data(x_e[i], y_e[i])
    pt_m.set_data(x_m[i], y_m[i])
    return ln_ship, pt_ship, pt_e, pt_m

ani = FuncAnimation(figA, update_anim, frames=N_FRAMES,
                    init_func=init_anim, interval=80, blit=True)

HTML(ani.to_jshtml())
